# 03 - Baseline, Track B Scoring, and H3 Export

This notebook combines the baseline layer and Track B nightlife layer, aggregates grid scores to Uber H3 resolution 8, and exports the final GeoJSON used by the web application.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
WEB_DATA = ROOT / "web" / "data"
print(ROOT)

## Run H3 Scoring

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(ROOT / "src" / "03_scoring_h3.py")], check=True)

## Inspect Scores

In [ ]:
h3_scores = pd.read_csv(PROCESSED / "h3_scores_track_b.csv")
metadata = json.loads((WEB_DATA / "metadata.json").read_text(encoding="utf-8"))
print(metadata["score_summary"])
display(h3_scores.head(10))
h3_scores[["baseline_score", "track_score", "composite_score"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
h3_scores["composite_score"].hist(bins=40, ax=ax)
ax.set_title("Composite H3 Score Distribution")
ax.set_xlabel("Score")
ax.set_ylabel("H3 cells")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plot_data = h3_scores.sample(min(15000, len(h3_scores)), random_state=1)
sc = ax.scatter(plot_data["lon"], plot_data["lat"], c=plot_data["composite_score"], s=3, cmap="viridis", alpha=0.75)
fig.colorbar(sc, ax=ax, label="Composite score")
ax.set_title("Track B H3 Scores - Shanghai")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="box")
plt.show()

## Weighting Rationale

Baseline score is the average of six universal needs using walk and bike scores only. Track B is a weighted nightlife layer: restaurants 30%, bars/nightlife 25%, cultural venues 20%, convenience stores 15%, and metro late proxy 10%. The final composite is 60% baseline and 40% track layer. Car mode scores are exported for comparison in the application but are not used in the main 15-minute-city score.

## Exported Files

- `data/processed/h3_scores_track_b.csv`: full tabular H3 scores.
- `data/processed/shanghai_15mc_h3_track_b.geojson`: complete H3 GeoJSON.
- `web/data/hexes.geojson`: compact front-end payload.
- `web/data/metadata.json`: source, scoring, and limitation metadata for the transparency panel.